<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/08_Test_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08 - Test Inference and Submission

Applies the frozen pipeline to the test split:

```
test_s1/s2/s3_cleaned.tsv
        |
        v
frozen blocking (exact name + name token + address token, generic country)
        |
        v
compute base + complementary features (same code as notebook 06)
        |
        v
load pairwise_model.joblib -> P(match)
        |
        v
load decision_config.json -> per-S1 match sets
        |
        v
matching_results.tsv + candidate_pairs.tsv
```

Country handling is fully generic: token frequency thresholds are recomputed on
the test sources themselves, so newly introduced countries (e.g. France) work
without any hard-coded country list.

**Do not run before 06 and 07 are finalised.**


In [ ]:
!pip install -q rapidfuzz indic-transliteration

import os
import re
import gc
import json
import random
import hashlib

import numpy as np
import pandas as pd

import joblib
import warnings

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit  # noqa: F401
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    fbeta_score,
)

warnings.filterwarnings("ignore")

print("imports ok")


In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"

DATASET_ROOT = os.path.join(
    DRIVE_ROOT, "01_Dataset", "6ab10eb3b23ba_student_resource",
    "student_resource", "dataset"
)
TEST_ROOT = os.path.join(DATASET_ROOT, "test")

CLEANED_DATA_ROOT = os.path.join(DRIVE_ROOT, "03_Experiments", "Cleaned_Data")
MODEL_ROOT = os.path.join(DRIVE_ROOT, "03_Experiments", "Model")

# Final submission folder. Mirrors the required package layout: output/.
OUTPUT_ROOT = os.path.join(DRIVE_ROOT, "05_Outputs", "Submission", "output")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

S1_PATH = os.path.join(CLEANED_DATA_ROOT, "test_s1_cleaned.tsv")
S2_PATH = os.path.join(CLEANED_DATA_ROOT, "test_s2_cleaned.tsv")
S3_PATH = os.path.join(CLEANED_DATA_ROOT, "test_s3_cleaned.tsv")

MODEL_PATH = os.path.join(MODEL_ROOT, "pairwise_model.joblib")
DECISION_PATH = os.path.join(MODEL_ROOT, "decision_config.json")

CANDIDATES_OUT = os.path.join(OUTPUT_ROOT, "candidate_pairs.tsv")
MATCHES_OUT = os.path.join(OUTPUT_ROOT, "matching_results.tsv")

print("model     :", os.path.exists(MODEL_PATH))
print("decision  :", os.path.exists(DECISION_PATH))
print("output dir:", OUTPUT_ROOT)


In [ ]:
MIN_TOKEN_FREQ = 2
MAX_TOKEN_FREQ = 500
CHUNK_SIZE = 200_000
FEATURE_BATCH = 200_000

NAME_STOPWORDS = {
    "inc", "incorporated", "llc", "ltd", "limited",
    "pvt", "private", "company", "co", "corp",
    "corporation", "group", "services", "partners",
    "holdings", "associates", "center", "india",
    "and", "of",
}


def meaningful_tokens(name):
    if name is None or pd.isna(name) or not str(name).strip():
        return []
    return [t for t in str(name).split() if t not in NAME_STOPWORDS and len(t) >= 2]


def allowed_from_counter(counter):
    return {tok for tok, c in counter.items() if MIN_TOKEN_FREQ <= c <= MAX_TOKEN_FREQ}


## Shared feature library (identical to notebook 06)

In [ ]:
import os
import re
import gc
import json
import random
import hashlib
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from rapidfuzz import fuzz


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_text(v):
    if v is None:
        return ""
    try:
        if pd.isna(v):
            return ""
    except (TypeError, ValueError):
        pass
    return str(v).strip()


# ============================================================
# SIMILARITY FEATURES (base)
# ============================================================

def char_similarity(a, b):
    a = clean_text(a)
    b = clean_text(b)
    if not a or not b:
        return np.nan
    return fuzz.ratio(a, b) / 100.0


def token_jaccard(a, b):
    A = set(clean_text(a).split())
    B = set(clean_text(b).split())
    if not A or not B:
        return np.nan
    return len(A & B) / len(A | B)


def ngram_jaccard(a, b, n=3):
    a = clean_text(a).replace(" ", "")
    b = clean_text(b).replace(" ", "")
    if not a or not b:
        return np.nan
    A = {a[i:i + n] for i in range(max(1, len(a) - n + 1))}
    B = {b[i:i + n] for i in range(max(1, len(b) - n + 1))}
    if not A or not B:
        return np.nan
    return len(A & B) / len(A | B)


def jaccard_sets(A, B):
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)


# ============================================================
# SINGLE-PASS TRANSLITERATION
# ============================================================

try:
    from indic_transliteration import sanscript
    from indic_transliteration.sanscript import transliterate
    HAS_TRANSLIT = True
except Exception:
    HAS_TRANSLIT = False


SCRIPT_MAP = {}
if HAS_TRANSLIT:
    SCRIPT_MAP = {
        "DEVANAGARI": sanscript.DEVANAGARI,
        "BENGALI": sanscript.BENGALI,
        "GURMUKHI": sanscript.GURMUKHI,
        "GUJARATI": sanscript.GUJARATI,
        "ORIYA": sanscript.ORIYA,
        "TAMIL": sanscript.TAMIL,
        "TELUGU": sanscript.TELUGU,
        "KANNADA": sanscript.KANNADA,
        "MALAYALAM": sanscript.MALAYALAM,
    }


def detect_script(text):
    # Returns the first non-Latin Indic script found, else None.
    if pd.isna(text):
        return None
    for ch in str(text):
        try:
            name = unicodedata.name(ch)
        except ValueError:
            continue
        for script_name in SCRIPT_MAP:
            if script_name in name:
                return script_name
    return None


def transliterate_text(text):
    # Transliterate exactly ONCE using the detected source script.
    if pd.isna(text):
        return ""
    text = str(text).strip()
    if not text:
        return ""
    if not HAS_TRANSLIT:
        return text.lower()
    script = detect_script(text)
    if script is None:
        return text.lower()
    try:
        return transliterate(text, SCRIPT_MAP[script], sanscript.ITRANS)
    except Exception:
        return text.lower()


def normalize_transliterated(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_transliteration(text):
    # Kept as the single public entry point used everywhere.
    return normalize_transliterated(transliterate_text(text))


_TRANSLIT_CACHE = {}


def get_translit(name):
    name = clean_text(name)
    key = name
    if key in _TRANSLIT_CACHE:
        return _TRANSLIT_CACHE[key]
    value = normalize_transliteration(name)
    _TRANSLIT_CACHE[key] = value
    return value


# ============================================================
# LEGAL-SUFFIX CORE NAME
# ============================================================

CORE_SUFFIXES = {
    "ltd", "limited", "inc", "incorporated", "corp", "corporation",
    "llc", "llp", "plc", "pvt", "private", "co", "company",
    "org", "organization",
}


CORE_SUFFIX_PATTERN = (
    r"(\s+\b("
    + "|".join(sorted((re.escape(s) for s in CORE_SUFFIXES), key=len, reverse=True))
    + r")\b)+$"
)


def create_name_core(name_series):
    # Strip ALL trailing legal suffixes, not just one:
    # "green logistics pvt ltd" -> "green logistics".
    result = pd.Series(name_series, dtype="string").fillna("").str.strip()
    result = result.str.replace(CORE_SUFFIX_PATTERN, "", regex=True)
    return result.str.replace(r"\s+", " ", regex=True).str.strip()


def core_of(name):
    result = clean_text(name)
    result = re.sub(CORE_SUFFIX_PATTERN, "", result)
    return re.sub(r"\s+", " ", result).strip()


# ============================================================
# PAIR FEATURE BUILDER
# ============================================================

NUMBER_RE = re.compile(r"\d+")


def numbers_of(text):
    return set(NUMBER_RE.findall(clean_text(text)))


def length_ratio(a, b):
    a = clean_text(a)
    b = clean_text(b)
    if not a or not b:
        return 0.0
    return min(len(a), len(b)) / max(len(a), len(b))


def token_count(text):
    return len(clean_text(text).split())


# Ordered list of the feature columns produced by compute_pair_features.
EXTRA_FEATURE_COLUMNS = [
    "name_exact",
    "name_core_exact",
    "name_length_ratio",
    "name_token_count_diff",
    "name_first_token_match",
    "name_last_token_match",
    "name_contains",
    "address_exact",
    "address_length_ratio",
    "address_number_match",
    "address_number_jaccard",
    "name_translit_exact",
    "name_translit_core_exact",
    "name_translit_char_similarity",
    "name_translit_3gram_jaccard",
    "name_translit_token_jaccard",
    "source_is_s2",
    "source_is_s3",
]

BASE_FEATURE_COLUMNS = [
    "name_char_similarity",
    "name_3gram_jaccard",
    "name_token_jaccard",
    "address_char_similarity",
    "address_token_jaccard",
    "country_exact",
]

ALL_FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + EXTRA_FEATURE_COLUMNS


def compute_pair_features(s1, cand, source):
    # s1 / cand are tuples of (country_norm, name_norm, name_core, address_norm).
    s1_country, s1_name, s1_core, s1_addr = s1
    c_country, c_name, c_core, c_addr = cand

    s1_name = clean_text(s1_name)
    c_name = clean_text(c_name)
    s1_addr = clean_text(s1_addr)
    c_addr = clean_text(c_addr)

    s1_translit = get_translit(s1_name)
    c_translit = get_translit(c_name)
    s1_translit_core = core_of(s1_translit)
    c_translit_core = core_of(c_translit)

    s1_tokens = s1_name.split()
    c_tokens = c_name.split()

    s1_nums = numbers_of(s1_addr)
    c_nums = numbers_of(c_addr)

    source = (source or "").upper()

    return {
        # ---- base features (recomputed so train/test use one code path) ----
        "name_char_similarity": char_similarity(s1_name, c_name),
        "name_3gram_jaccard": ngram_jaccard(s1_name, c_name, 3),
        "name_token_jaccard": token_jaccard(s1_name, c_name),
        "address_char_similarity": char_similarity(s1_addr, c_addr),
        "address_token_jaccard": token_jaccard(s1_addr, c_addr),
        "country_exact": int(clean_text(s1_country) == clean_text(c_country)),
        # ---- complementary name features ----
        "name_exact": int(s1_name != "" and s1_name == c_name),
        "name_core_exact": int(clean_text(s1_core) != "" and clean_text(s1_core) == clean_text(c_core)),
        "name_length_ratio": length_ratio(s1_name, c_name),
        "name_token_count_diff": abs(token_count(s1_name) - token_count(c_name)),
        "name_first_token_match": int(bool(s1_tokens) and bool(c_tokens) and s1_tokens[0] == c_tokens[0]),
        "name_last_token_match": int(bool(s1_tokens) and bool(c_tokens) and s1_tokens[-1] == c_tokens[-1]),
        "name_contains": int(
            s1_name != "" and c_name != "" and s1_name != c_name
            and (s1_name in c_name or c_name in s1_name)
        ),
        # ---- complementary address features ----
        "address_exact": int(s1_addr != "" and s1_addr == c_addr),
        "address_length_ratio": length_ratio(s1_addr, c_addr),
        "address_number_match": int(bool(s1_nums & c_nums)),
        "address_number_jaccard": jaccard_sets(s1_nums, c_nums),
        # ---- transliteration features ----
        "name_translit_exact": int(s1_translit != "" and s1_translit == c_translit),
        "name_translit_core_exact": int(s1_translit_core != "" and s1_translit_core == c_translit_core),
        "name_translit_char_similarity": char_similarity(s1_translit, c_translit),
        "name_translit_3gram_jaccard": ngram_jaccard(s1_translit, c_translit, 3),
        "name_translit_token_jaccard": token_jaccard(s1_translit, c_translit),
        # ---- source indicators ----
        "source_is_s2": int(source == "S2"),
        "source_is_s3": int(source == "S3"),
    }


# ============================================================
# ID -> TEXT LOOKUPS
# ============================================================

def build_text_lookup(path, id_col="entity_id"):
    lookup = {}
    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=[id_col, "country_norm", "name_norm", "name_core", "address_norm"],
        chunksize=200_000,
        dtype="string",
    ):
        for row in chunk.itertuples(index=False):
            lookup[getattr(row, id_col)] = (
                row.country_norm,
                row.name_norm,
                row.name_core,
                row.address_norm,
            )
        del chunk
        gc.collect()
    return lookup


def build_gt_lookup(path):
    gt = pd.read_csv(
        path,
        sep="\t",
        dtype={"source1_entity_id": "string", "matched_entity_ids": "string"},
    )
    lookup = {}
    for row in gt.itertuples(index=False):
        s1_id = row.source1_entity_id
        matched = row.matched_entity_ids
        if pd.isna(matched) or matched == "":
            lookup[s1_id] = set()
        else:
            lookup[s1_id] = {x.strip() for x in str(matched).split(",") if x.strip()}
    del gt
    gc.collect()
    return lookup


# ============================================================
# DETERMINISTIC S1 HASH SPLIT
# ============================================================

def stable_bucket(text, mod):
    h = hashlib.md5(str(text).encode("utf-8")).hexdigest()
    return int(h[:8], 16) % mod


def is_validation_entity(s1_id, mod, rem):
    return stable_bucket(s1_id, mod) < rem


## Frozen blocking

In [ ]:
# ------------------------------------------------------------------
# Frozen blocking. Country is used only as an exact key, never as a
# hard-coded list, so unseen test countries work unchanged.
# ------------------------------------------------------------------

def build_name_index(path):
    index = defaultdict(list)
    for chunk in pd.read_csv(
        path, sep="\t",
        usecols=["entity_id", "country_norm", "name_core"],
        chunksize=CHUNK_SIZE, dtype="string",
    ):
        for eid, country, name in zip(chunk["entity_id"], chunk["country_norm"], chunk["name_core"]):
            if pd.isna(name) or str(name).strip() == "":
                continue
            index[(country, name)].append(eid)
        del chunk
        gc.collect()
    return index


def build_token_index(path, column, allowed, tokenizer):
    index = defaultdict(list)
    for chunk in pd.read_csv(
        path, sep="\t",
        usecols=["entity_id", column],
        chunksize=CHUNK_SIZE, dtype="string",
    ):
        for eid, value in zip(chunk["entity_id"], chunk[column]):
            if pd.isna(value):
                continue
            for token in set(tokenizer(value)):
                if token in allowed:
                    index[token].append(eid)
        del chunk
        gc.collect()
    return index


def token_counter(path, column, tokenizer):
    counter = Counter()
    for chunk in pd.read_csv(
        path, sep="\t",
        usecols=[column],
        chunksize=CHUNK_SIZE, dtype="string",
    ):
        for value in chunk[column].dropna():
            counter.update(set(tokenizer(value)))
        del chunk
        gc.collect()
    return counter


def iter_candidates(s1_path, name_index, name_token_index, address_token_index,
                    allowed_name_tokens, allowed_address_tokens, source_tag):
    for chunk in pd.read_csv(
        s1_path, sep="\t",
        usecols=["entity_id", "country_norm", "name_core", "address_norm"],
        chunksize=CHUNK_SIZE, dtype="string",
    ):
        for eid, country, name, address in zip(
            chunk["entity_id"], chunk["country_norm"],
            chunk["name_core"], chunk["address_norm"],
        ):
            cands = set()
            if not pd.isna(name) and str(name).strip():
                cands.update(name_index.get((country, name), []))
                for token in set(meaningful_tokens(name)):
                    if token in allowed_name_tokens:
                        cands.update(name_token_index.get(token, []))
            if not pd.isna(address) and str(address).strip():
                for token in set(str(address).split()):
                    if token in allowed_address_tokens:
                        cands.update(address_token_index.get(token, []))
            for cand_id in cands:
                yield eid, cand_id, source_tag
        del chunk
        gc.collect()


In [ ]:
# Build blocking structures per source so peak memory stays bounded.
SOURCE_SPECS = [("S2", S2_PATH), ("S3", S3_PATH)]
BLOCKERS = {}

for tag, path in SOURCE_SPECS:
    print("building blocker:", tag)
    name_counter = token_counter(path, "name_core", meaningful_tokens)
    addr_counter = token_counter(path, "address_norm", lambda a: str(a).split())
    allowed_name = allowed_from_counter(name_counter)
    allowed_addr = allowed_from_counter(addr_counter)
    BLOCKERS[tag] = {
        "name_index": build_name_index(path),
        "name_token_index": build_token_index(path, "name_core", allowed_name, meaningful_tokens),
        "address_token_index": build_token_index(path, "address_norm", allowed_addr, lambda a: str(a).split()),
        "allowed_name_tokens": allowed_name,
        "allowed_address_tokens": allowed_addr,
    }
    print("  blocks:", len(BLOCKERS[tag]["name_index"]),
          len(BLOCKERS[tag]["name_token_index"]),
          len(BLOCKERS[tag]["address_token_index"]))


## Load model and decision configuration

In [ ]:
model = joblib.load(MODEL_PATH)
with open(DECISION_PATH, "r", encoding="utf-8") as f:
    DECISION = json.load(f)

THRESHOLD = float(DECISION["threshold"])
RELATIVE = float(DECISION.get("relative", 0.0))

with open(os.path.join(MODEL_ROOT, "feature_columns.json"), "r", encoding="utf-8") as f:
    FEATURE_NAMES = json.load(f)

print("threshold:", THRESHOLD, "| relative:", RELATIVE)
print("features :", len(FEATURE_NAMES))


## Score every candidate pair

In [ ]:
S1_TEXT = build_text_lookup(S1_PATH, id_col="entity_id")
print("test S1 records:", len(S1_TEXT))

pred_frames = []

for tag, _ in SOURCE_SPECS:
    b = BLOCKERS[tag]
    source_text = build_text_lookup(
        dict(SOURCE_SPECS)[tag], id_col="entity_id"
    )
    print(tag, "records:", len(source_text))

    batch_ids, batch_cand, batch_src, batch_X = [], [], [], []

    def flush():
        if not batch_X:
            return
        Xb = np.nan_to_num(np.asarray(batch_X, dtype="float32"), nan=0.0)
        probs = model.predict_proba(Xb)[:, 1]
        pred_frames.append(pd.DataFrame({
            "s1_entity_id": batch_ids,
            "candidate_entity_id": batch_cand,
            "source": batch_src,
            "match_probability": probs,
        }))
        batch_ids.clear(); batch_cand.clear(); batch_src.clear(); batch_X.clear()

    for eid, cand_id, src in iter_candidates(
        S1_PATH,
        b["name_index"], b["name_token_index"], b["address_token_index"],
        b["allowed_name_tokens"], b["allowed_address_tokens"], tag,
    ):
        s1 = S1_TEXT.get(eid)
        cand = source_text.get(cand_id)
        if s1 is None or cand is None:
            continue
        feats = compute_pair_features(s1, cand, src)
        batch_ids.append(eid)
        batch_cand.append(cand_id)
        batch_src.append(src)
        batch_X.append([feats[c] for c in FEATURE_NAMES])
        if len(batch_X) >= FEATURE_BATCH:
            flush()

    flush()
    del source_text
    gc.collect()

candidate_pairs = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame(
    columns=["s1_entity_id", "candidate_entity_id", "source", "match_probability"]
)
print("test candidate pairs:", len(candidate_pairs))


## Apply decision layer and write submission

In [ ]:
# ------------------------------------------------------------------
# Submission format (challenge requirements)
#
#   candidate_pairs.tsv : source1_entity_id, candidate_entity_ids
#                         one row per test S1, comma separated, no duplicates
#   matching_results.tsv: source1_entity_id, matched_entity_ids
#                         the final matches (a subset of the candidates)
#
# Entities with no candidates / no match get an empty list.
# ------------------------------------------------------------------

# Every test S1 entity, in file order -> exactly one row each.
s1_ids = pd.read_csv(
    S1_PATH, sep="\t", usecols=["entity_id"], dtype="string"
)["entity_id"].tolist()

groups = {k: v for k, v in candidate_pairs.groupby("s1_entity_id", sort=False)}

candidate_rows = []
matched_rows = []

for s1_id in s1_ids:
    group = groups.get(s1_id)

    if group is None or len(group) == 0:
        candidate_rows.append({"source1_entity_id": s1_id, "candidate_entity_ids": ""})
        matched_rows.append({"source1_entity_id": s1_id, "matched_entity_ids": ""})
        continue

    # Highest probability first; dedupe while preserving that order.
    group = group.sort_values("match_probability", ascending=False)
    cand_ids = list(dict.fromkeys(group["candidate_entity_id"].tolist()))

    probs = group["match_probability"].to_numpy()
    cut = max(THRESHOLD, RELATIVE * float(probs.max()))
    matched = list(
        dict.fromkeys(
            group.loc[probs >= cut, "candidate_entity_id"].tolist()
        )
    )
    # Guarantee matches are a subset of candidates.
    cand_set = set(cand_ids)
    matched = [c for c in matched if c in cand_set]

    candidate_rows.append(
        {"source1_entity_id": s1_id, "candidate_entity_ids": ",".join(cand_ids)}
    )
    matched_rows.append(
        {"source1_entity_id": s1_id, "matched_entity_ids": ",".join(matched)}
    )

candidate_out = pd.DataFrame(candidate_rows)
matches_out = pd.DataFrame(matched_rows)

candidate_out.to_csv(CANDIDATES_OUT, sep="\t", index=False)
matches_out.to_csv(MATCHES_OUT, sep="\t", index=False)

print("wrote", CANDIDATES_OUT)
print("wrote", MATCHES_OUT)
print("test S1 entities      :", len(matches_out))
print("non-empty predictions :", (matches_out["matched_entity_ids"] != "").sum())
print("total candidates      :", candidate_out["candidate_entity_ids"].str.count(",").sum()
      + (candidate_out["candidate_entity_ids"] != "").sum())
print(matches_out.head())


## Validate the submission format

In [ ]:
# ------------------------------------------------------------------
# Local format check (mirrors the official utils/validate_submission.py).
# The official script is authoritative; run it too:
#   python3 utils/validate_submission.py \
#       --matching output/matching_results.tsv \
#       --candidate output/candidate_pairs.tsv \
#       --test-dir dataset/test
# ------------------------------------------------------------------

valid_ids = set(
    pd.read_csv(S2_PATH, sep="\t", usecols=["entity_id"], dtype="string")["entity_id"]
) | set(
    pd.read_csv(S3_PATH, sep="\t", usecols=["entity_id"], dtype="string")["entity_id"]
)

issues = []
if matches_out["source1_entity_id"].duplicated().any():
    issues.append("duplicate source1_entity_id rows in matching_results.tsv")
if candidate_out["source1_entity_id"].duplicated().any():
    issues.append("duplicate source1_entity_id rows in candidate_pairs.tsv")
if set(matches_out["source1_entity_id"]) != set(s1_ids):
    issues.append("matching_results.tsv does not cover every test S1 entity")
if set(candidate_out["source1_entity_id"]) != set(s1_ids):
    issues.append("candidate_pairs.tsv does not cover every test S1 entity")

for table, column in [
    (matches_out, "matched_entity_ids"),
    (candidate_out, "candidate_entity_ids"),
]:
    for s1, value in zip(table["source1_entity_id"], table[column]):
        if not value:
            continue
        parts = value.split(",")
        if len(parts) != len(set(parts)):
            issues.append(f"{s1}: duplicate ids in {column}")
        for pid in parts:
            if not (pid.startswith("S2-") or pid.startswith("S3-")):
                issues.append(f"{s1}: '{pid}' is not an S2-/S3- id")
            elif pid not in valid_ids:
                issues.append(f"{s1}: '{pid}' does not exist in the test set")

cand_sets = dict(
    zip(candidate_out["source1_entity_id"], candidate_out["candidate_entity_ids"])
)
for s1, value in zip(matches_out["source1_entity_id"], matches_out["matched_entity_ids"]):
    if not value:
        continue
    cset = set(p for p in cand_sets.get(s1, "").split(",") if p)
    for pid in value.split(","):
        if pid and pid not in cset:
            issues.append(f"{s1}: match '{pid}' not present in candidates")

if issues:
    print("FAIL - fix before submitting:")
    for i, issue in enumerate(issues[:50], 1):
        print(i, issue)
else:
    print("PASS - submission files satisfy the format rules")
